<a href="https://colab.research.google.com/github/vincentmartin/tp-rag-student-version/blob/main/mini-projet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-projet : une équipe d'agents de veille scientifique

L'énoncé complet (attendus, livrables, barème) est dans le [README](https://github.com/vincentmartin/tp-rag-student-version#mini-projet--une-équipe-dagents-de-veille-scientifique).

Ce notebook vous donne l'infrastructure, pour que votre temps aille à la **conception de l'équipe d'agents** et à son **évaluation** :

| Partie | Contenu | Statut |
|---|---|---|
| 0 | Installation, données | fourni |
| 1 | Modèles, et vos fonctions du TP | à coller |
| 2 | Trois corpus indexés : articles, brevets, bibliographies | fourni |
| 3 | Les outils de chaque spécialiste | fourni |
| 4 | L'équipe d'agents | **à concevoir** |
| 5 | Évaluation : citations, couverture, coût | outils fournis, protocole à écrire |

Exécution sur Colab avec un GPU T4, comme le TP (`Exécution -> Modifier le type d'exécution -> T4 GPU`).

## 0. Installation et données

In [ ]:
%pip install -q -U transformers accelerate sentence-transformers chromadb rank_bm25 pypdf langgraph

In [ ]:
import difflib
import html
import operator
import re
import time
import uuid
from collections import Counter
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import chromadb
import numpy as np
import pandas as pd
import torch
from pydantic import BaseModel, Field, ValidationError
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

assert torch.cuda.is_available(), "Pas de GPU : vérifiez le type d'exécution (T4 GPU)."
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
pd.set_option("display.max_colwidth", 300)
set_seed(42)

In [ ]:
DATA = Path("data")
if not (DATA / "patents").exists():  # sur Colab ou Kaggle
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/vincentmartin/tp-rag-student-version.git
    !git -C tp-rag-student-version sparse-checkout set data/arxiv data/patents data/latex
    DATA = Path("tp-rag-student-version/data")

pdf_paths = sorted((DATA / "arxiv").glob("*.pdf"))
patent_paths = sorted((DATA / "patents").glob("*.txt"))
bib_paths = sorted((DATA / "latex").glob("*.bib"))
print(len(pdf_paths), "articles,", len(patent_paths), "brevets,", len(bib_paths), "bibliographies")
assert pdf_paths and patent_paths and bib_paths, "Corpus introuvable : vérifiez que le clonage a réussi."

## 1. Modèles et fonctions du TP

Les modèles sont ceux du TP. `generate` est la même, à un ajout près : chaque appel est **enregistré** dans `USAGE` avec le nom de l'agent appelant (`agent="superviseur"`, par exemple). C'est ce qui vous permettra de mesurer ce que coûte chaque rôle. Passez toujours `agent=...`.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="cuda")
embedder = SentenceTransformer(
    "google/embeddinggemma-2", device="cuda",
    model_kwargs={"dtype": torch.float32}, config_kwargs={"vision_config": None, "audio_config": None},
)
reranker = CrossEncoder("Qwen/Qwen3-Reranker-0.6B", device="cuda", model_kwargs={"dtype": DTYPE})

USAGE = []  # un enregistrement par appel au LLM


@torch.no_grad()
def generate(messages, max_new_tokens=300, tools=None, enable_thinking=False, agent="?", **gen_kwargs):
    """Génère la réponse de l'assistant (décodage glouton par défaut) et enregistre son coût dans USAGE."""
    start = time.time()
    inputs = tokenizer.apply_chat_template(
        messages, tools=tools, enable_thinking=enable_thinking, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    gen_kwargs = {"do_sample": False, "temperature": None, "top_p": None, "top_k": None, **gen_kwargs}
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
    new_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    USAGE.append({
        "agent": agent, "prompt_tokens": inputs["input_ids"].shape[1], "new_tokens": len(new_ids), "seconds": time.time() - start,
    })
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def generate_structured(messages, schema, max_retries=3, agent="?"):
    """Génère une réponse JSON validée par le modèle Pydantic `schema`."""
    fields = "\n".join(f"- {n} ({f.annotation.__name__}) : {f.description}" for n, f in schema.model_fields.items())
    instruction = f"\n\nRéponds uniquement avec un objet JSON contenant les champs :\n{fields}"
    messages = messages[:-1] + [{**messages[-1], "content": messages[-1]["content"] + instruction}]
    for attempt in range(max_retries):
        raw = generate(messages, agent=agent, **({"do_sample": True, "temperature": 0.7} if attempt else {}))
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        try:
            return schema.model_validate_json(match.group(0) if match else raw)
        except ValidationError:
            pass
    raise ValueError(f"Pas de JSON valide après {max_retries} tentatives : {raw}")


def usage_summary(records=None):
    """Coût par agent : nombre d'appels, tokens lus et générés, secondes."""
    frame = pd.DataFrame(records if records is not None else USAGE)
    if frame.empty:
        return frame
    summary = frame.groupby("agent").agg(
        appels=("seconds", "size"), tokens_lus=("prompt_tokens", "sum"), tokens_générés=("new_tokens", "sum"), secondes=("seconds", "sum"),
    )
    return summary.sort_values("secondes", ascending=False).round(1)

Fonctions **fournies** dans le TP (extraction, découpage, index, recherche vectorielle) :

In [ ]:
def load_pages(path):
    pages = []
    for number, page in enumerate(PdfReader(path).pages, start=1):
        text = re.sub(r"(\w)-\n(\w)", r"\1\2", page.extract_text() or "")
        pages.append({"source": path.stem, "page": number, "text": re.sub(r"\s+", " ", text).strip()})
    return pages


def chunk_text(text, size=1000, overlap=200):
    chunks, current = [], []
    for word in text.split():
        if current and len(" ".join(current + [word])) > size:
            chunks.append(" ".join(current))
            while current and (len(" ".join(current)) > overlap or len(" ".join(current + [word])) > size):
                current.pop(0)
        current.append(word)
    return chunks + ([" ".join(current)] if current else [])


chroma = chromadb.EphemeralClient()


def build_collection(name, chunks, titles, batch_size=2000):
    """Indexe des chunks `{"id", "text", métadonnées...}` ; `titles` donne le titre de chaque chunk."""
    if name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(name)
    collection = chroma.create_collection(name, configuration={"hnsw": {"space": "cosine"}})
    for start in range(0, len(chunks), batch_size):
        batch, batch_titles = chunks[start : start + batch_size], titles[start : start + batch_size]
        collection.add(
            ids=[c["id"] for c in batch],
            embeddings=embed_passages([c["text"] for c in batch], batch_titles),
            documents=[c["text"] for c in batch],
            metadatas=[{k: v for k, v in c.items() if k not in ("id", "text")} for c in batch],
        )
    return collection


def vector_search(query, k=5, where=None, index=None):
    index = collection if index is None else index
    result = index.query(query_embeddings=[embed_query(query)], n_results=k, where=where)
    rows = zip(result["ids"][0], result["documents"][0], result["metadatas"][0], result["distances"][0])
    return [{"id": i, "text": text, "score": 1 - distance, **meta} for i, text, meta, distance in rows]


def format_context(results):
    return "\n\n".join(f"[{i}] ({c['source']}, page {c['page']})\n{c['text']}" for i, c in enumerate(results, start=1))

**À faire** : collez ci-dessous **vos** solutions du TP :
- `embed_passages`, `embed_query` (exercice 1) ;
- `rerank_search` (exercice 4) ;
- `search_articles`, `list_articles`, `read_page` (exercice 6) ;
- `parse_tool_calls` (exercice 7) ;
- `execute`, **modifiée** pour prendre le dictionnaire d'outils en paramètre : `execute(call, tools)`. Chaque agent aura ses propres outils.

In [ ]:
# VOTRE CODE ICI

## 2. Les trois corpus

### 2.1 Articles

L'index du TP : 10 synthèses arXiv, découpées en chunks de 1000 caractères.

In [ ]:
TITLES = {path.stem: path.stem.replace("_", " ") for path in pdf_paths}
pages = [page for path in pdf_paths for page in load_pages(path)]
chunks = [
    {"id": f"{p['source']}-p{p['page']}-{i}", "source": p["source"], "page": p["page"], "text": text}
    for p in pages for i, text in enumerate(chunk_text(p["text"]))
]
collection = build_collection("articles", chunks, [TITLES[c["source"]] for c in chunks])
print(len(pages), "pages,", collection.count(), "chunks")

### 2.2 Brevets

25 brevets européens (EP, publiés en 2000) : un titre, une description et des revendications. Les fichiers sources sont des tableaux contenant du HTML ; certains brevets n'ont pas de description, un seul n'a que son titre.

In [ ]:
def strip_tags(text):
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", text))).strip()


def load_patent(path):
    patent = {"id": None, "date": None, "title": path.stem, "description": "", "claims": []}
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split("\t")
        if len(parts) < 8:
            continue
        _, number, _, date, _, kind, _, text = parts[:8]
        patent["id"], patent["date"] = f"EP{number}", date
        if kind == "TITLE":
            patent["title"] = text.strip()
        elif kind == "DESCR":
            patent["description"] += strip_tags(text)
        elif kind == "CLAIM":
            patent["claims"] += [strip_tags(c) for c in re.findall(r"<claim-text>(.*?)</claim>", text, re.DOTALL)]
    return patent


PATENTS = {p["id"]: p for p in map(load_patent, patent_paths)}
patent_chunks = []
for p in PATENTS.values():
    for i, text in enumerate(chunk_text(p["description"])):
        patent_chunks.append({"id": f"{p['id']}-d{i}", "source": p["id"], "page": 0, "part": "description", "text": text})
    for n, claim in enumerate(p["claims"], start=1):
        for i, text in enumerate(chunk_text(claim)):
            patent_chunks.append({"id": f"{p['id']}-c{n}-{i}", "source": p["id"], "page": n, "part": "revendication", "text": text})
patent_index = build_collection("brevets", patent_chunks, [PATENTS[c["source"]]["title"] for c in patent_chunks])
print(len(PATENTS), "brevets,", patent_index.count(), "chunks")

### 2.3 Bibliographies

Les sources LaTeX de 9 des 10 articles (pas celui sur la pêche) : chaque entrée `.bib`, avec le nombre de fois où l'article la cite (`\cite{...}` dans le `.tex`). Les `.bib` sont hétérogènes : certains n'ont qu'un champ `title` qui contient toute la référence.

In [ ]:
def clean_latex(text):
    text = re.sub(r"\\[a-zA-Z]+\*?", " ", text)
    return re.sub(r"\s+", " ", re.sub(r"[{}~\\$]", " ", text)).strip()


def bib_field(body, name):
    match = re.search(rf"\b{name}\s*=\s*[{{\"](.+?)[}}\"]\s*,?\s*\n", body + "\n", re.IGNORECASE | re.DOTALL)
    return clean_latex(match.group(1)) if match else ""


def find_article(name):
    """Identifiant d'article le plus proche d'un nom approximatif (fragment, titre...)."""
    norm = lambda s: re.sub(r"[^a-z0-9]+", " ", s.lower().replace("&", "and")).strip()
    exact = [a for a in TITLES if norm(name) and norm(name) in norm(a)]
    if len(exact) == 1:
        return exact[0]
    best = max(TITLES, key=lambda a: difflib.SequenceMatcher(None, norm(name), norm(a)).ratio())
    return best if difflib.SequenceMatcher(None, norm(name), norm(best)).ratio() > 0.5 else None


def load_bibliography(bib_path):
    article = find_article(bib_path.stem)
    tex_path = bib_path.with_suffix(".tex")
    tex = tex_path.read_text(encoding="utf-8", errors="ignore") if tex_path.exists() else ""
    cited = Counter(
        key.strip() for keys in re.findall(r"\\\w*cite\w*\*?(?:\[[^\]]*\])*\{([^}]*)\}", tex) for key in keys.split(",")
    )
    text = bib_path.read_text(encoding="utf-8", errors="ignore")
    entries = []
    for match in re.finditer(r"^[ \t]*@(\w+)\s*\{\s*([^,\s]+)\s*,(.*?)(?=^[ \t]*@\w+\s*\{|\Z)", text, re.MULTILINE | re.DOTALL):
        kind, key, body = match.groups()
        if kind.lower() in {"string", "comment", "preamble"}:
            continue
        year = bib_field(body, "year") or next(iter(re.findall(r"\b(19\d\d|20\d\d)\b", body)[-1:]), "")
        entries.append({
            "article": article, "key": key, "title": bib_field(body, "title"), "author": bib_field(body, "author"),
            "year": int(year[:4]) if year[:4].isdigit() else None, "cited": cited.get(key, 0),
        })
    return entries


REFERENCES = pd.DataFrame([entry for path in bib_paths for entry in load_bibliography(path)])
tokenize = lambda text: re.findall(r"\w+", str(text).lower())
ref_bm25 = BM25Okapi([tokenize(f"{r.title} {r.author} {r.key}") for r in REFERENCES.itertuples()])
REFERENCES.groupby("article").agg(entrées=("key", "size"), citées=("cited", lambda c: (c > 0).sum()))

## 3. Les outils des spécialistes

Trois boîtes à outils, une par spécialiste. Comme dans le TP, la docstring est **tout** ce que voit le modèle. Vous pouvez les modifier, en ajouter (et le justifier dans le rapport).

In [ ]:
ARTICLE_TOOLS = {f.__name__: f for f in (search_articles, list_articles, read_page)}


def find_patent(name):
    name = name.strip()
    if name.upper().replace(" ", "") in PATENTS:
        return name.upper().replace(" ", "")
    matches = [i for i, p in PATENTS.items() if name.lower() in p["title"].lower()]
    return matches[0] if len(matches) == 1 else None


def search_patents(query: str, patent: str = "") -> str:
    """Recherche des extraits de brevets européens (en anglais) sur UN fait précis.

    Args:
        query: La question ou les mots-clés à rechercher, de préférence en anglais.
        patent: Numéro du brevet (par exemple EP0608223) pour restreindre la recherche (facultatif).
    """
    where = None
    if patent:
        found = find_patent(patent)
        if found is None:
            return f"Brevet inconnu : {patent}. Utilise list_patents."
        where = {"source": found}
    results = vector_search(query, 4, where=where, index=patent_index)
    return "\n\n".join(
        f"[{i}] ({c['source']}, {c['part']}{' ' + str(c['page']) if c['page'] else ''})\n{c['text']}" for i, c in enumerate(results, start=1)
    )


def list_patents() -> str:
    """Liste les brevets du corpus : numéro, date de publication, titre, nombre de revendications."""
    return "\n".join(f"- {p['id']} ({p['date']}) : {p['title']} [{len(p['claims'])} revendications]" for p in PATENTS.values())


def read_claims(patent: str, first: str = "1", last: str = "3") -> str:
    """Lit les revendications d'un brevet (ce qu'il protège). La revendication 1 est la plus générale.

    Args:
        patent: Numéro du brevet (par exemple EP0608223) ou fragment de son titre.
        first: Numéro de la première revendication à lire.
        last: Numéro de la dernière revendication à lire.
    """
    found = find_patent(patent)
    if found is None:
        return f"Brevet inconnu : {patent}. Utilise list_patents."
    claims = PATENTS[found]["claims"][int(first) - 1 : int(last)]
    if not claims:
        return f"Aucune revendication {first}-{last} pour {found} ({len(PATENTS[found]['claims'])} au total)."
    return "\n\n".join(f"Revendication {n} : {c}" for n, c in enumerate(claims, start=int(first)))


PATENT_TOOLS = {f.__name__: f for f in (search_patents, list_patents, read_claims)}


def format_references(rows):
    return "\n".join(
        f"- {r.article} / {r.key} ({int(r.year) if pd.notna(r.year) else 'année ?'}) : {r.title[:200]} ; {r.author[:80] or 'auteurs ?'} [cité {r.cited} fois]"
        for r in rows.itertuples()
    )


def search_references(query: str, article: str = "") -> str:
    """Recherche des références bibliographiques (titre, auteurs) citées par les articles du corpus.

    Args:
        query: Mots-clés (titre, nom d'auteur, sujet), en anglais.
        article: Identifiant ou fragment du titre de l'article dont on cherche les références (facultatif).
    """
    scores = pd.Series(ref_bm25.get_scores(tokenize(query)), index=REFERENCES.index)
    if article:
        found = find_article(article)
        if found is None:
            return f"Article inconnu : {article}. Utilise list_articles."
        scores = scores[REFERENCES["article"] == found]
    best = scores[scores > 0].sort_values(ascending=False).index[:5]
    return format_references(REFERENCES.loc[best]) if len(best) else "Aucune référence trouvée."


def most_cited(article: str, n: str = "5") -> str:
    """Donne les références les plus souvent citées dans le texte d'un article.

    Args:
        article: Identifiant ou fragment du titre de l'article.
        n: Nombre de références à renvoyer.
    """
    found = find_article(article)
    rows = REFERENCES[REFERENCES["article"] == found]
    if rows.empty:
        return f"Pas de bibliographie disponible pour : {article}."
    return format_references(rows.sort_values("cited", ascending=False).head(int(n)))


def reference_stats(article: str) -> str:
    """Statistiques sur la bibliographie d'un article : nombre de références, références citées, années.

    Args:
        article: Identifiant ou fragment du titre de l'article.
    """
    found = find_article(article)
    rows = REFERENCES[REFERENCES["article"] == found]
    if rows.empty:
        return f"Pas de bibliographie disponible pour : {article}."
    years = rows["year"].dropna()
    return (
        f"{found} : {len(rows)} références, dont {(rows['cited'] > 0).sum()} citées dans le texte. "
        f"Années : {int(years.min())}-{int(years.max())}, médiane {int(years.median())}."
    )


BIB_TOOLS = {f.__name__: f for f in (search_references, most_cited, reference_stats)}

# Démonstration
print(read_claims("EP0615422", "1", "1")[:300], "\n")
print(most_cited("Multi-Agent", "3"), "\n")
print(reference_stats("quantum states"), "\n")
print(search_patents("measurement of blood pressure")[:500])

## 4. Votre équipe d'agents

C'est le cœur du projet. Le README fixe l'équipe minimale (superviseur, trois spécialistes, rédacteur, relecteur) et les comparaisons attendues. Quelques repères :

- **Un agent = un rôle** : un prompt système, des outils (ou aucun), et une **vue** de l'état. Le rédacteur, par exemple, ne voit que ce que les spécialistes ont rapporté. Un spécialiste peut être une boucle comme celle du TP (un sous-graphe LangGraph) avec ses propres outils.
- **Les transmissions** : décidez ce qu'un spécialiste renvoie (extraits bruts, résumé, faits structurés en Pydantic avec leur citation). C'est un choix à mesurer.
- **La délégation** : le superviseur peut produire une sortie structurée (`generate_structured`), par exemple `{"agent": ..., "tâche": ...}`. Pour plusieurs tâches indépendantes, voyez l'API `Send` de LangGraph (sur un seul GPU, les appels restent séquentiels : qu'est-ce que la parallélisation apporte alors ?).
- **Les garde-fous** : budget global d'appels, nombre maximal de délégations, au plus une révision demandée par le relecteur.
- **Le format des citations** dans la note finale, imposé pour l'évaluation automatique : `[article:<identifiant>, p. <page>]`, `[brevet:<numéro>]`, `[réf:<article>/<clé>]`. Les identifiants peuvent être approximatifs (fragment du titre).
- **La référence** : l'agent unique du TP, avec les neuf outils réunis.

Passez `agent="<rôle>"` à chaque appel de `generate`, sinon le coût par rôle sera inexploitable.

In [ ]:
# Votre équipe d'agents

## 5. Évaluation

### 5.1 Vos questions

Trois exemples du format attendu. Écrivez-en **8 à 10**, qui obligent à croiser les corpus, avec les faits qu'une bonne note doit contenir (vérifiés par vous dans les documents).

In [ ]:
QUESTIONS = [
    {
        "question": "Le corpus contient-il à la fois un article et un brevet sur la mesure de la pression dans le corps humain ? Comparez ce qu'ils mesurent et comment.",
        "sources": ["article:Blood_Pressure", "brevet:EP0615422"],
        "facts": ["La synthèse porte sur la mesure de la pression artérielle et ses biais", "Le brevet EP0615422 porte sur un appareil de mesure de la pression auriculaire (atrial pressure)"],
    },
    {
        "question": "Quelle est la référence la plus citée par la synthèse sur l'apprentissage par renforcement multi-agent, et quel rôle joue-t-elle dans la synthèse ?",
        "sources": ["réf:Multi-Agent", "article:Multi-Agent"],
        "facts": ["Human-level control through deep reinforcement learning (DQN), Mnih et al., 2015", "Elle est citée 26 fois"],
    },
    {
        "question": "Quels brevets du corpus relèvent de la biologie moléculaire ou du génie génétique ?",
        "sources": ["brevet:EP0600893", "brevet:EP0600877", "brevet:EP0668908", "brevet:EP0636181", "brevet:EP0600865"],
        "facts": ["Expression de gènes chez les bacilles (EP0600893)", "Sélection de ribozymes (EP0600877)", "Séquences d'ADN de nématode pour le diagnostic (EP0668908)", "Gène de l'oxalate oxydase chez les plantes (EP0636181)", "Production de xylanase (EP0600865)"],
    },
]

### 5.2 Outils d'évaluation (fournis)

- `extract_citations(note)` : les phrases de la note et leurs citations ;
- `citation_precision(note)` : pour chaque citation, le juge vérifie que le texte cité appuie la phrase ;
- `fact_coverage(note, facts)` : proportion des faits attendus présents dans la note.

Le juge est le même modèle que vos agents : relisez ses verdicts, et discutez-en la fiabilité dans le rapport.

In [ ]:
CITATION = re.compile(r"\[(article|brevet|réf)\s*:\s*([^\]]+)\]")


class Support(BaseModel):
    reason: str = Field(description="Justification en une phrase")
    supported: bool = Field(description="true si le texte source appuie l'affirmation")


class Presence(BaseModel):
    present: bool = Field(description="true si la note contient cette information, même formulée autrement")


def extract_citations(note):
    """Liste de (phrase, type, cible) pour chaque citation de la note."""
    found = []
    for sentence in re.split(r"(?<=[.!?])\s+(?![^\[]*\])|\n+", note):  # sans couper à l'intérieur d'une citation
        for kind, target in CITATION.findall(sentence):
            found.append((re.sub(r"\s+([.,;!?])", r"\1", CITATION.sub("", sentence)).strip(), kind, target.strip()))
    return found


def cited_text(kind, target, claim):
    """Le texte auquel renvoie une citation (ou None si elle ne désigne rien)."""
    if kind == "article":
        name, _, page = target.partition(", p.")
        source = find_article(name)
        if source is None:
            return None
        if page.strip().isdigit():
            return next((p["text"] for p in pages if p["source"] == source and p["page"] == int(page)), None)
        return format_context(vector_search(claim, 3, where={"source": source}))
    if kind == "brevet":
        patent = find_patent(target.split(",")[0])
        return None if patent is None else "\n\n".join(c["text"] for c in vector_search(claim, 3, where={"source": patent}, index=patent_index))
    article, _, key = target.partition("/")
    rows = REFERENCES[(REFERENCES["article"] == find_article(article)) & (REFERENCES["key"] == key.strip())]
    return None if rows.empty else format_references(rows)


def citation_precision(note):
    """Proportion des citations dont la source appuie la phrase ; détail par citation."""
    rows = []
    for claim, kind, target in extract_citations(note):
        source = cited_text(kind, target, claim)
        if source is None:
            rows.append({"phrase": claim, "citation": f"{kind}:{target}", "supported": False, "reason": "source introuvable"})
            continue
        verdict = generate_structured([{"role": "user", "content": (
            f"Texte source :\n{source[:6000]}\n\nAffirmation : {claim}\n\nLe texte source appuie-t-il cette affirmation ?"
        )}], Support, agent="juge")
        rows.append({"phrase": claim, "citation": f"{kind}:{target}", "supported": verdict.supported, "reason": verdict.reason})
    detail = pd.DataFrame(rows, columns=["phrase", "citation", "supported", "reason"])
    return (detail["supported"].mean() if len(detail) else 0.0), detail


def fact_coverage(note, facts):
    present = [
        generate_structured([{"role": "user", "content": f"Note :\n{note}\n\nInformation : {fact}\n\nLa note contient-elle cette information ?"}],
                            Presence, agent="juge").present
        for fact in facts
    ]
    return float(np.mean(present)) if facts else float("nan")


# Démonstration sur une note écrite à la main
demo = ("Le brevet EP0615422 décrit un appareil de mesure de la pression auriculaire [brevet:EP0615422]. "
        "La synthèse sur la pression artérielle a été publiée en 1850 [article:Blood_Pressure, p. 1].")
precision, detail = citation_precision(demo)
print(f"précision des citations : {precision:.2f}", "| couverture :", fact_coverage(demo, QUESTIONS[0]["facts"]))
detail

### 5.3 Votre protocole

Pour chaque système (agent unique du TP, vos architectures, l'ablation sans relecteur), sur toutes vos questions : précision des citations, couverture des faits, sources consultées, et coût par rôle (`usage_summary`, après `USAGE.clear()`). Rangez tout dans un DataFrame, puis comparez par système et par type de question.

In [ ]:
# Votre protocole d'évaluation